# 🎬 Die Evolution des Kinos — Teil 2: Einnahmen-Prognose

Trainiert drei Regressionsmodelle auf ~6.000 Filmen mit Budget- und Einnahmedaten, erklärt das beste Modell mit SHAP und prognostiziert die Einnahmen der erwarteten 2026-Blockbuster.

**Input:** `data/movies_financials.csv`, `data/movies_2026_upcoming.csv` (aus Teil 1)
**Output:** `models/` (Modell + `model_meta.json`), `data/predictions_2026.csv`, Grafiken in `images/`

In [ ]:
# Nur nötig in Colab — lokal: pip install -r requirements.txt
%pip install -q xgboost shap

In [ ]:
# ── Setup: läuft in Google Colab UND lokal ───────────────────────────────────
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Projektordner: lokal = Repo-Root (auch wenn aus notebooks/ gestartet), Colab = /content
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR  = ROOT / "data"
MODEL_DIR = ROOT / "models"
IMG_DIR   = ROOT / "images"
RAW_DIR   = ROOT / "kaggle_data"          # Rohdaten von Kaggle (nicht im Repo)
for d in (DATA_DIR, MODEL_DIR, IMG_DIR, RAW_DIR):
    d.mkdir(parents=True, exist_ok=True)

def get_secret(name):
    """Colab: aus Colab Secrets (🔑 links). Lokal: aus Umgebungsvariable / .env."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            pass
    return os.environ.get(name)

print(f"{'Colab' if IN_COLAB else 'Lokal'} | Projektordner: {ROOT}")

In [ ]:
import gzip
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import xgboost as xgb
import shap

sns.set_theme(style='whitegrid', palette='muted')
print('✅ All imports successful')

## CSV aus Teil 1 laden

In [ ]:
needed = ["movies_financials.csv", "movies_2026_upcoming.csv"]
missing = [f for f in needed if not (DATA_DIR / f).exists()]

if missing and IN_COLAB:
    # In Colab hat jedes Notebook eine eigene Laufzeit → Dateien aus Teil 1 hochladen
    from google.colab import files
    print(f"Bitte hochladen: {', '.join(missing)}  (aus teil1_output.zip → data/)")
    for name in files.upload():
        target = next((f for f in needed if name.startswith(f.replace('.csv', ''))), name)
        os.replace(name, DATA_DIR / target)
elif missing:
    raise FileNotFoundError(f"{missing} fehlen in {DATA_DIR} — zuerst Teil 1 ausführen.")

fin_df      = pd.read_csv(DATA_DIR / 'movies_financials.csv')
upcoming_df = pd.read_csv(DATA_DIR / 'movies_2026_upcoming.csv')

print(f'Financial dataset : {fin_df.shape}')
print(f'Upcoming 2026     : {upcoming_df.shape}')
fin_df.head(3)

## Verteilung der Einnahmen

In [ ]:
#  3.1  Distribution of revenue (log scale)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(fin_df['revenue_adj'] / 1e6, bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Revenue Distribution (raw, $M adj.)')
axes[0].set_xlabel('Revenue ($M)')

axes[1].hist(np.log1p(fin_df['revenue_adj']), bins=60, color='salmon', edgecolor='white')
axes[1].set_title('Revenue Distribution (log scale)')
axes[1].set_xlabel('log(Revenue)')

plt.tight_layout()
plt.savefig(IMG_DIR / 'revenue_distribution.png', dpi=150)
plt.show()

Die rohen Daten sind stark rechtsschief — wenige Blockbuster wie Avatar verzerren alles. Deshalb transformieren wir die Zielvariable mit dem natürlichen Logarithmus. Das Modell lernt dann auf einer gleichmäßigen Skala und macht für alle Budgetklassen bessere Vorhersagen.

## Korrelationsmatrix

In [ ]:
# ── 3.2  Correlation heatmap ──────────────────────────────────────────────────
numeric_cols = ['budget_adj', 'revenue_adj', 'runtime', 'vote_average',
                'vote_count', 'popularity', 'release_month', 'roi']
corr = fin_df[numeric_cols].dropna().corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix – Financial Features', fontweight='bold')
plt.tight_layout()
plt.savefig(IMG_DIR / 'correlation_heatmap.png', dpi=150)
plt.show()

Welche Variablen haben den stärksten Zusammenhang mit den Einnahmen? Budget und Vote-Count korrelieren am stärksten. Vote-Count kennt man aber erst **nach** dem Kinostart — deshalb ist es bewusst **kein** Modell-Feature (sonst wäre es Data Leakage). Interessant ist auch, dass ein höheres Budget den ROI leicht senkt — teure Filme sind also riskanter, nicht zwingend profitabler.

In [ ]:
# ── 3.3  Median revenue by genre ──────────────────────────────────────────────
genre_rev = (
    fin_df.groupby('primary_genre')['revenue_adj']
    .agg(['median', 'count'])
    .query('count >= 20')
    .sort_values('median', ascending=True)
)

plt.figure(figsize=(9, 6))
plt.barh(genre_rev.index, genre_rev['median'] / 1e6, color='mediumpurple')
plt.xlabel('Median Revenue ($M, inflation-adjusted)')
plt.title('Median Box Office Revenue by Genre', fontweight='bold')
plt.tight_layout()
plt.savefig(IMG_DIR / 'revenue_by_genre.png', dpi=150)
plt.show()

Animation und Adventure führen deutlich — Familienfilme sind extrem profitabel. Interessant ist Horror: trotz kleiner Budgets landet es im Mittelfeld, was auf einen sehr guten ROI hindeutet.

In [ ]:
#  3.4  Budget vs Revenue scatter
sample = fin_df.sample(min(3000, len(fin_df)), random_state=42)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(
    np.log1p(sample['budget_adj']),
    np.log1p(sample['revenue_adj']),
    c=sample['is_oscar_winner'].astype(int),
    cmap='RdYlGn', alpha=0.5, s=20
)
plt.colorbar(scatter, label='Oscar Winner (1=Yes)')
plt.xlabel('log(Budget adj.)')
plt.ylabel('log(Revenue adj.)')
plt.title('Budget vs Revenue (log scale) — Oscar winners in green', fontweight='bold')
plt.tight_layout()
plt.savefig(IMG_DIR / 'budget_vs_revenue.png', dpi=150)
plt.show()

## Feature Engineering

In [ ]:
# ── 4.1  Select and encode features ──────────────────────────────────────────
FEATURE_COLS = [
    'log_budget',        # main financial driver
    'runtime',           # film length
    'release_month',     # seasonality
    'is_summer',         # peak season flag
    'is_holiday',        # holiday season flag
    'genre_encoded',     # encoded genre
    'is_oscar_winner',   # prestige
    'release_year',      # temporal trend
]
TARGET = 'log_revenue'

# Genre-Encoder immer hier neu fitten: derselbe Encoder wird für Training,
# 2026-Prognosen und das Dashboard benutzt → identische Codes überall.
le_genre = LabelEncoder()
fin_df['primary_genre'] = fin_df['primary_genre'].fillna('Unknown')
fin_df['genre_encoded'] = le_genre.fit_transform(fin_df['primary_genre'])

fin_df['log_budget']      = np.log1p(fin_df['budget_adj'])
fin_df['log_revenue']     = np.log1p(fin_df['revenue_adj'])
fin_df['is_summer']       = fin_df['release_month'].isin([5,6,7,8]).astype(int)
fin_df['is_holiday']      = fin_df['release_month'].isin([11,12]).astype(int)
fin_df['is_oscar_winner'] = fin_df['is_oscar_winner'].astype(bool).astype(int)

# Drop rows with missing features
model_df = fin_df[FEATURE_COLS + [TARGET]].dropna()
print(f'Model dataset: {model_df.shape}')
model_df.describe()

In [ ]:
# 4.2  Train / Test split
X = model_df[FEATURE_COLS]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## Modelle trainieren und vergleichen

In [ ]:
#  5.1  Define models
models = {
    'Random Forest': RandomForestRegressor( # 300 Bäume stimmen gleichzeitig ab — robust, erklärbar, und zeigt welche Features am wichtigsten sind
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=4,
        n_jobs=-1,
        random_state=42
    ),
    'XGBoost': xgb.XGBRegressor( # Jeder Baum korrigiert den vorherigen — aktuell einer der stärksten Algorithmen für strukturierte Daten
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    ),
    'Gradient Boosting': GradientBoostingRegressor( # Das ältere Prinzip hinter XGBoost — als Vergleichsmodell eingebaut
        n_estimators=200,
        max_depth=5,
        learning_rate=0.08,
        random_state=42
    )
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2  = r2_score(y_test, y_pred)
    mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred)) / 1e6
    rmse= np.sqrt(mean_squared_error(y_test, y_pred))

    results[name] = {'R²': r2, 'MAE ($M)': mae, 'RMSE (log)': rmse}
    print(f'{name:<22}  R²={r2:.3f}  MAE=${mae:.1f}M  RMSE(log)={rmse:.3f}')

results_df = pd.DataFrame(results).T.sort_values('R²', ascending=False)
print('\n', results_df)

**Cross-Validation:** Die Daten sind nach Erscheinungsjahr sortiert. Ohne Mischen (`shuffle`) bestünde ein Fold fast nur aus alten Filmen der 1920er–1980er, die das Modell kaum gesehen hat — das ergibt einen irreführend negativen R². Deshalb werden die Folds gemischt.

In [ ]:
# ── 5.2  Cross-validation on best model ───────────────────────────────────────
best_name = results_df.index[0]
best_model = models[best_name]

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X, y, cv=cv, scoring='r2', n_jobs=-1)
print(f'Best model    : {best_name}')
print(f'CV R² scores  : {cv_scores.round(3)}')
print(f'Mean CV R²    : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

In [ ]:
# ── 5.3  Model comparison chart ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(results_df.index, results_df['R²'], color=['gold','steelblue','salmon'])
axes[0].set_title('R² Score by Model (higher = better)')
axes[0].set_ylim(0, 1)
axes[0].axhline(0.6, color='gray', linestyle='--', alpha=0.5, label='0.6 threshold')
axes[0].legend()

axes[1].bar(results_df.index, results_df['MAE ($M)'], color=['gold','steelblue','salmon'])
axes[1].set_title('Mean Absolute Error ($M) — lower = better')
axes[1].set_ylabel('MAE ($M)')

plt.tight_layout()
plt.savefig(IMG_DIR / 'model_comparison.png', dpi=150)
plt.show()

**Schwaches R².** Das Modell kennt nur Budget, Genre, Timing und Laufzeit. Es fehlen die eigentlichen Treiber des Kassenerfolgs: Star Power, Marketing-Budget, Franchise/Sequel, Konkurrenz am Startwochenende, Kritiken und Mundpropaganda.

In [ ]:
# ── 6.1  Predicted vs Actual ──────────────────────────────────────────────────
y_pred_best = best_model.predict(X_test)

# Convert back from log scale to dollars
actual_rev  = np.expm1(y_test) / 1e6
pred_rev    = np.expm1(y_pred_best) / 1e6

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: predicted vs actual (log scale)
axes[0].scatter(y_test, y_pred_best, alpha=0.3, s=15, color='steelblue')
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual log(Revenue)')
axes[0].set_ylabel('Predicted log(Revenue)')
axes[0].set_title(f'{best_name} – Predicted vs Actual (log)')
axes[0].legend()

# Residuals distribution
residuals = y_test - y_pred_best
axes[1].hist(residuals, bins=50, color='salmon', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_xlabel('Residual (log scale)')
axes[1].set_title('Residuals Distribution')

plt.suptitle(f'{best_name} Model Evaluation', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(IMG_DIR / 'model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'R²  : {r2_score(y_test, y_pred_best):.3f}')
print(f'MAE : ${mean_absolute_error(actual_rev, pred_rev):.1f}M')
median_ae_M = float(np.median(np.abs(actual_rev - pred_rev)))
print(f'Median absolute error: ${median_ae_M:.1f}M')

# 80%-Prognoseintervall aus den echten Fehlern auf dem Testset (funktioniert für jedes Modell):
# 10% der Testfilme lagen weiter darunter, 10% weiter darüber.
INTERVAL_LOG = (float(np.percentile(residuals, 10)), float(np.percentile(residuals, 90)))
coverage = np.mean((residuals >= INTERVAL_LOG[0]) & (residuals <= INTERVAL_LOG[1]))
print(f'80% interval (log): {INTERVAL_LOG[0]:+.2f} … {INTERVAL_LOG[1]:+.2f}  '
      f'→ ×{np.exp(INTERVAL_LOG[0]):.2f} … ×{np.exp(INTERVAL_LOG[1]):.2f} of the forecast  (coverage {coverage:.0%})')

In [ ]:
# ── 6.2  Feature Importance ───────────────────────────────────────────────────
FEATURE_LABELS = {
    'log_budget':      'Production Budget (log)',
    'runtime':         'Runtime (minutes)',
    'release_month':   'Release Month',
    'is_summer':       'Summer Release',
    'is_holiday':      'Holiday Release',
    'genre_encoded':   'Genre',
    'is_oscar_winner': 'Oscar Winner',
    'release_year':    'Release Year',
}

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
else:
    importances = np.zeros(len(FEATURE_COLS))

imp_df = pd.DataFrame({
    'feature': [FEATURE_LABELS.get(f, f) for f in FEATURE_COLS],
    'importance': importances
}).sort_values('importance')

plt.figure(figsize=(9, 5))
colors = ['#2ecc71' if i == imp_df['importance'].idxmax() else 'steelblue'
          for i in imp_df.index]
plt.barh(imp_df['feature'], imp_df['importance'], color=colors)
plt.xlabel('Feature Importance (impurity-based)')
plt.title(f'Feature Importance – {best_name}', fontweight='bold')
plt.tight_layout()
plt.savefig(IMG_DIR / 'feature_importance.png', dpi=150)
plt.show()

print('Top feature:', imp_df.sort_values('importance', ascending=False).iloc[0]['feature'])

In [ ]:
# ── 6.3  SHAP values (model explainability) ───────────────────────────────────
# SHAP shows HOW each feature pushes the prediction up or down
print('Computing SHAP values (may take ~30s)…')

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test.head(500))  # sample for speed

plt.figure(figsize=(9, 5))
shap.summary_plot(
    shap_values,
    X_test.head(500),
    feature_names=[FEATURE_LABELS.get(f, f) for f in FEATURE_COLS],
    show=False
)
plt.title('SHAP Feature Impact on Revenue Prediction', fontweight='bold')
plt.tight_layout()
plt.savefig(IMG_DIR / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ Red = high feature value pushes revenue UP, Blue = down')

## Prognose für die 2026-Blockbuster

In [ ]:
# ── 7.1  Prepare upcoming movies features ─────────────────────────────────────
df_pred = upcoming_df.copy()

# Apply same transformations as training data
if 'budget_adj' not in df_pred.columns:
    df_pred['budget_adj'] = df_pred['budget'] * 1.05  # 2026 CPI multiplier

df_pred['log_budget']     = np.log1p(df_pred['budget_adj'])
df_pred['is_summer']      = df_pred['release_month'].isin([5,6,7,8]).astype(int)
df_pred['is_holiday']     = df_pred['release_month'].isin([11,12]).astype(int)
df_pred['is_oscar_winner']= 0  # upcoming films haven't won yet
df_pred['release_year']   = 2026

# Encode genres — handle unseen labels gracefully
known_classes = set(le_genre.classes_)
df_pred['genre_safe'] = df_pred['primary_genre'].apply(
    lambda g: g if g in known_classes else 'Drama'  # fallback
)
df_pred['genre_encoded'] = le_genre.transform(df_pred['genre_safe'])

# Fill missing runtime with median from training set
median_runtime = float(fin_df['runtime'].median())
df_pred['runtime'] = df_pred['runtime'].fillna(median_runtime)

X_pred = df_pred[FEATURE_COLS]
print('Features ready for prediction:')
X_pred

In [ ]:
# ── 7.2  Generate predictions with confidence intervals ───────────────────────
log_preds = best_model.predict(X_pred)
rev_preds = np.expm1(log_preds) / 1e6  # convert to $M

# 80% interval from the test-set errors (see 6.1)
low_80  = np.expm1(log_preds + INTERVAL_LOG[0]) / 1e6
high_80 = np.expm1(log_preds + INTERVAL_LOG[1]) / 1e6

predictions_df = pd.DataFrame({
    'Movie':             df_pred['title'],
    'Budget ($M)':       (df_pred['budget_adj'] / 1e6).round(0),
    'Genre':             df_pred['primary_genre'],
    'Release Month':     df_pred['release_month'],
    'Predicted Rev ($M)': rev_preds.round(0),
    'Low Estimate ($M)': low_80.round(0),
    'High Estimate ($M)':high_80.round(0),
    'Predicted ROI (%)': ((rev_preds - df_pred['budget_adj'].values/1e6)
                          / (df_pred['budget_adj'].values/1e6) * 100).round(1)
})

print('\n🎬 2026 Box Office Revenue Predictions:\n')
print(predictions_df.to_string(index=False))

In [ ]:
# ── 7.3  Prediction chart with confidence intervals ───────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

movies   = predictions_df['Movie']
pred_rev = predictions_df['Predicted Rev ($M)'].values
low_err  = (pred_rev - predictions_df['Low Estimate ($M)'].values)
high_err = (predictions_df['High Estimate ($M)'].values - pred_rev)

ax.barh(movies, pred_rev, color='steelblue', zorder=2)
ax.errorbar(
    pred_rev, range(len(movies)),
    xerr=[low_err, high_err],
    fmt='none', color='black', capsize=5, linewidth=1.5, zorder=3
)

# Annotate bars
for i, (rev, roi) in enumerate(zip(pred_rev, predictions_df['Predicted ROI (%)'])):
    ax.text(rev + 10, i, f'${rev:.0f}M  (ROI: {roi:.0f}%)',
            va='center', fontsize=9)

ax.set_xlabel('Predicted Worldwide Revenue ($M)')
ax.set_title(f'2026 Blockbuster Revenue Predictions ({best_name})\nwith 80% prediction interval',
             fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(0, predictions_df['High Estimate ($M)'].max() * 1.15)

plt.tight_layout()
plt.savefig(IMG_DIR / 'predictions_2026.png', dpi=150)
plt.show()

## Modell speichern

In [ ]:
# ── 8.1  Save model + metadata ────────────────────────────────────────────────
# XGBoost → its own JSON format, which loads with any newer xgboost version.
# (A pickle only loads with the exact library versions it was saved with.)
for old in MODEL_DIR.glob('film_revenue_model.*'):
    old.unlink()
if best_name == 'XGBoost':
    model_file = 'film_revenue_model.json'
    best_model.save_model(MODEL_DIR / model_file)
else:
    model_file = 'film_revenue_model.pkl.gz'
    with gzip.open(MODEL_DIR / model_file, 'wb') as f:
        pickle.dump(best_model, f)

meta = {
    'model_name':     best_name,
    'model_file':     model_file,
    'feature_cols':   FEATURE_COLS,
    'feature_labels': FEATURE_LABELS,
    'genres':         list(le_genre.classes_),   # index = genre_encoded
    'median_runtime': median_runtime,
    'interval_log':   INTERVAL_LOG,               # 80% prediction interval on log scale
    'trained_on':     len(X_train),
    'test_r2':        round(r2_score(y_test, y_pred_best), 4),
    'cv_r2_mean':     round(float(cv_scores.mean()), 4),
    'cv_r2_std':      round(float(cv_scores.std()), 4),
    'test_mae_M':     round(mean_absolute_error(np.expm1(y_test)/1e6, np.expm1(y_pred_best)/1e6), 1),
    'test_median_ae_M': round(median_ae_M, 1),
    'model_comparison': results_df.round(4).reset_index(names='model').to_dict(orient='records'),
    'versions': {'python': sys.version.split()[0], 'scikit-learn': sklearn.__version__,
                 'numpy': np.__version__, 'pandas': pd.__version__, 'xgboost': xgb.__version__},
}
(MODEL_DIR / 'model_meta.json').write_text(json.dumps(meta, indent=2))

# Self-check: reload exactly like app.py does and compare predictions
if model_file.endswith('.json'):
    reloaded = xgb.XGBRegressor(); reloaded.load_model(MODEL_DIR / model_file)
else:
    with gzip.open(MODEL_DIR / model_file, 'rb') as f:
        reloaded = pickle.load(f)
assert np.allclose(reloaded.predict(X_pred), best_model.predict(X_pred), atol=1e-4)

# ── 8.2  Save predictions CSV ────────────────────────────────────────────────
predictions_df.to_csv(DATA_DIR / 'predictions_2026.csv', index=False)

size_mb = (MODEL_DIR / model_file).stat().st_size / 1e6
print(f'✅ Model saved  → models/{model_file} ({size_mb:.1f} MB) — reload check passed')
print(f'   Model type   : {best_name}')
print(f'   Test R²      : {meta["test_r2"]}  |  CV R²: {meta["cv_r2_mean"]} ± {meta["cv_r2_std"]}')
print(f'   Test MAE     : ${meta["test_mae_M"]}M  |  median error: ${meta["test_median_ae_M"]}M')
print(f'✅ Predictions  → data/predictions_2026.csv')

## Ergebnisse herunterladen (nur Colab)

Lädt Modell, Prognosen und Grafiken als ZIP herunter.

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    out = Path("/content/teil2_export")
    if out.exists():
        shutil.rmtree(out)
    (out / "data").mkdir(parents=True)
    shutil.copytree(MODEL_DIR, out / "models")
    shutil.copytree(IMG_DIR, out / "images")
    shutil.copy(DATA_DIR / "predictions_2026.csv", out / "data")
    shutil.make_archive("/content/teil2_output", "zip", out)
    files.download("/content/teil2_output.zip")
else:
    print("Lokal: Dateien liegen bereits in models/, data/ und images/")